In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install --upgrade transformers accelerate tokenizers scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 117.7 MB/s eta 0:00:00


In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

save_path = "/content/drive/MyDrive/insight-agent/models/distilbert_sentiment_final"

tokenizer = DistilBertTokenizerFast.from_pretrained(save_path)
model = DistilBertForSequenceClassification.from_pretrained(save_path)

print("Model reloaded.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model reloaded.


In [ ]:
import pandas as pd

test = pd.read_csv("/content/drive/MyDrive/insight-agent/data/test.csv").fillna("")
print(test.shape)

(7500, 4)


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
print("Using device:", device)

Using device: cuda


In [ ]:
test_encodings = tokenizer(
    test["clean_content"].tolist(), padding=True, truncation=True, max_length=256, return_tensors="pt"
)
print(test_encodings["input_ids"].shape)

torch.Size([7500, 256])


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

batch_size = 32
input_ids = test_encodings["input_ids"]
attention_mask = test_encodings["attention_mask"]

dataset = TensorDataset(input_ids, attention_mask)
loader = DataLoader(dataset, batch_size=batch_size)

all_preds = []
all_probs = []

with torch.no_grad():
    for batch_input_ids, batch_attention_mask in loader:
        batch_input_ids = batch_input_ids.to(device)
        batch_attention_mask = batch_attention_mask.to(device)

        outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

print(f"Generated predictions for {len(all_preds)} examples")

Generated predictions for 7500 examples


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_true = test["label"].tolist()

print("Test Accuracy:", accuracy_score(y_true, all_preds))
print(classification_report(y_true, all_preds))

Test Accuracy: 0.9242666666666667
              precision    recall  f1-score   support

           0       0.90      0.96      0.93      3744
           1       0.95      0.89      0.92      3756

    accuracy                           0.92      7500
   macro avg       0.93      0.92      0.92      7500
weighted avg       0.93      0.92      0.92      7500



In [ ]:
cm = confusion_matrix(y_true, all_preds)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[3576  168]
 [ 400 3356]]


In [ ]:
comparison = {
    "TF-IDF + Logistic Regression (baseline)": 0.8517,   # your Day 4 number
    "Fine-tuned DistilBERT": accuracy_score(y_true, all_preds),
}
for model_name, acc in comparison.items():
    print(f"{model_name}: {acc:.4f}")

TF-IDF + Logistic Regression (baseline): 0.8517
Fine-tuned DistilBERT: 0.9243


In [ ]:
import json

results = {
    "baseline_test_accuracy": 0.8517,
    "distilbert_test_accuracy": float(accuracy_score(y_true, all_preds)),
    "confusion_matrix": cm.tolist(),
}

with open("/content/drive/MyDrive/insight-agent/models/eval_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved.")

Results saved.


In [ ]:
results_df = test.copy().reset_index(drop=True)
results_df["predicted"] = all_preds
results_df["confidence"] = [max(p) for p in all_probs]
results_df["correct"] = results_df["label"] == results_df["predicted"]

misclassified = results_df[results_df["correct"] == False]
print(f"Total misclassified: {len(misclassified)} out of {len(results_df)}")

sample_errors = misclassified.sample(5, random_state=1)
for _, row in sample_errors.iterrows():
    print(f"Actual: {row['label']} | Predicted: {row['predicted']} | Confidence: {row['confidence']:.3f}")
    print(f"Review: {row['clean_content'][:300]}")
    print("-" * 80)

Total misclassified: 568 out of 7500
Actual: 1 | Predicted: 0 | Confidence: 0.973
Review: fenriz you'd make purdy girl. fake female vocals aside this album is a must for darkthrone fans. it sounds like they took a bunch of acid and got lost in the woods. luckily they had instruments and a four track to document this demented and just plain BENT musical journey. this is by far my favorite
--------------------------------------------------------------------------------
Actual: 1 | Predicted: 0 | Confidence: 0.653
Review: If you don't like hooks and chorus and enjoy pure rap, this album's for you. This is an emcee not BSing w/ commercial ish. Pure lyrics. Due to complex lyrics, this album isn't meant for uneducated individuals. If you're smart and like rap, cop this album! Let the mind stimulation begin.
--------------------------------------------------------------------------------
Actual: 1 | Predicted: 0 | Confidence: 0.991
Review: I purchased this product in March and so far no coast

# In a text cell or just as a printed summary
print(f"""
Day 8 Results Summary
----------------------
Baseline (TF-IDF + LogReg) test accuracy: 85.17%
Fine-tuned DistilBERT test accuracy: {accuracy_score(y_true, all_preds)*100:.2f}%
Improvement: {(accuracy_score(y_true, all_preds) - 0.8517)*100:.2f} percentage points
Total misclassified (DistilBERT): {len(misclassified)} / {len(results_df)}
""")